In [1]:
from operator import add
from typing import TypedDict, Annotated

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langgraph.constants import START, END
from langgraph.graph import StateGraph, MessagesState

load_dotenv()


True

In [16]:
# 1.初始化模型
model = init_chat_model(
    model="deepseek-flash",
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    }
)

# 2.创建带历史消息的State
# class LLMState(TypedDict):
#     messages:Annotated[list[BaseMessage],add]

# 创建Node
def llM_node(state:MessagesState):
    # 获取历史消息，也包含最新一次的用户消息
    messages = state.get("messages")
    # 调用模型，传入messages消息历史，返回ai_message，类型是AIMessage
    response = model.invoke(messages)
    # print("==================")
    # print(response.text)
    # print("==================")

    # 写回state
    # return {"ai_message":response.text}
    return {"messages":[response]}

In [17]:
# 4.添加Edge，连接Node，形成Graph
builder = StateGraph(MessagesState)

builder.add_node("llM", llM_node)

builder.add_edge(START, "llM")
builder.add_edge("llM", END)

graph = builder.compile()

In [18]:
# 5.调用Graph
result = graph.invoke({"messages": [HumanMessage("你好，你是谁？")]})
print(result)

{'messages': [HumanMessage(content='你好，你是谁？', additional_kwargs={}, response_metadata={}, id='450a058a-4682-466a-8fc0-833d4079e7ae'), AIMessage(content='你好！很高兴认识你！😊\n\n我是 **DeepSeek**，由深度求索公司创造的AI助手。我支持文本和图片输入，擅长回答各种问题、提供帮助和进行对话。\n\n关于我的一些特点：\n- 🆓 **完全免费**使用\n- 📚 **知识截止日期**：2026年2月\n- 📖 **上下文长度**：1M tokens（可以一次性处理像《三体》三部曲那样的长篇内容）\n- 📎 **文件上传**：支持上传图片、PDF、Word、Excel、PPT等文件，并读取其中的文字信息\n- 🔗 **阅读链接**：可以阅读网页链接内容\n- 🌐 **联网搜索**：支持联网搜索（需要手动在Web/App端开启）\n- 🎙️ **语音输入**：App端支持语音输入\n\n有什么我可以帮你的吗？无论是学习、工作、生活问题，还是只是想聊聊天，我都很乐意陪你！✨', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 203, 'prompt_tokens': 8, 'total_tokens': 211, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe0164